In [ ]:
## This will be the project page for questions 2-6. Below I have inserteed my modular code segment that I submitted for homwork 5 last week. 

In [ ]:
## Modeling the angora 2D brace frame element

In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [4]:
def universal_truss_solver(nodes, elements, properties, bcs, forces):
    num_nodes = len(nodes)
    num_dof = 2 * num_nodes
    K_global = np.zeros((num_dof, num_dof))
    
    # 1. Global Stiffness Matrix Assembly
    for i, (n1, n2) in enumerate(elements):
        EA = properties[i]
        x1, y1 = nodes[n1]
        x2, y2 = nodes[n2]
        
        L = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)
        c = (x2 - x1) / L
        s = (y2 - y1) / L
        
        # Element stiffness matrix (Global)
        ke = (EA / L) * np.array([
            [c*c,  c*s, -c*c, -c*s],
            [c*s,  s*s, -c*s, -s*s],
            [-c*c, -c*s,  c*c,  c*s],
            [-c*s, -s*s,  c*s,  s*s]
        ])
        
        dofs = [2*n1, 2*n1+1, 2*n2, 2*n2+1]
        for ix, row in enumerate(dofs):
            for jx, col in enumerate(dofs):
                K_global[row, col] += ke[ix, jx]

    # 2. Setup Forces and Boundary Conditions
    F = np.zeros(num_dof)
    for node_idx, force_vec in forces.items():
        F[2*node_idx] = force_vec[0]
        F[2*node_idx+1] = force_vec[1]
        
    free_dofs = np.ones(num_dof, dtype=bool)
    for node_idx, fixed in bcs.items():
        if fixed[0] is not None: free_dofs[2*node_idx] = False
        if fixed[1] is not None: free_dofs[2*node_idx+1] = False
    
    # 3. Solve for Displacement Field
    U = np.zeros(num_dof)
    K_reduced = K_global[np.ix_(free_dofs, free_dofs)]
    F_reduced = F[free_dofs]
    U[free_dofs] = np.linalg.solve(K_reduced, F_reduced)
    
    # 4. Calculate Internal Forces for each Element
    member_forces = []
    U_reshaped = U.reshape(-1, 2)
    for i, (n1, n2) in enumerate(elements):
        EA = properties[i]
        x1, y1 = nodes[n1]
        x2, y2 = nodes[n2]
        L = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)
        c, s = (x2 - x1) / L, (y2 - y1) / L
        
        u_local = np.array([U_reshaped[n1][0], U_reshaped[n1][1], 
                           U_reshaped[n2][0], U_reshaped[n2][1]])
        T = np.array([-c, -s, c, s])
        force = (EA / L) * np.dot(T, u_local)
        member_forces.append(force)
        
    return U, member_forces

# --- TEST VARIATIONS HERE ---

E = 29e6  # psi
area_main = np.pi * (5**2)    # 10in diameter (r=5)
area_diag = np.pi * (1.5**2)  # 3in diameter (r=1.5)

# 1. Nodes (converted ft to inches)
nodes = np.array([
    [0, 0],          # 0: BL
    [0, 144],        # 1: TL
    [288, 0],        # 2: BR
    [288, 144],      # 3: TR
    [144, 72]        # 4: Center
])

# 2. Elements [Start, End]
elements = [
    [0, 1], [2, 3],  # Vertical supports
    [0, 4], [1, 4],  # Left diagonals
    [2, 4], [3, 4]   # Right diagonals
]

# 3. Match EA to elements
properties = [
    E * area_main, E * area_main, # Verticals
    E * area_diag, E * area_diag, # Diagonals
    E * area_diag, E * area_diag
]

# 4. Boundary Conditions (Fixed on the left, perhaps?)
# Let's assume the left side is bolted to a wall
bcs = {0: [0, 0], 1: [0, 0]}

# 5. Forces (Central load of 48k lbs downward)
forces = {4: [0, -48000]}

# RUN
U_result, P_result = universal_truss_solver(nodes, elements, properties, bcs, forces)

print("--- Displacements ---")
for i in range(len(U_result)//2):
    print(f"Node {i+1}: ux={U_result[2*i]:.4e}, uy={U_result[2*i+1]:.4e}")

print("\n--- Internal Axial Forces (P) ---")
for i, f in enumerate(P_result):
    state = "Tension" if f > 1e-9 else ("Compression" if f < -1e-9 else "Zero-Force")
    print(f"Element {i+1}: {f:.2f} ({state})")

--- Displacements ---
Node 1: ux=0.0000e+00, uy=0.0000e+00
Node 2: ux=0.0000e+00, uy=0.0000e+00
Node 3: ux=5.0217e-02, uy=6.1874e-03
Node 4: ux=-5.0217e-02, uy=6.1874e-03
Node 5: ux=-8.2041e-20, uy=-9.4247e-02

--- Internal Axial Forces (P) ---
Element 1: 0.00 (Zero-Force)
Element 2: 0.00 (Zero-Force)
Element 3: -53665.63 (Compression)
Element 4: 53665.63 (Tension)
Element 5: 0.00 (Zero-Force)
Element 6: 0.00 (Zero-Force)


In [ ]:
## Converting P1 into a lateral wind force at the top of the frame

In [ ]:
### This conversion need the tributary area of the building that each frame is supporting. So using google earth I have measure the buildings floor
### layout to be roughly 7400 sqft. This coumes out to 1480 sq ft of trivutary space for each of the estimated 5 frames supporting the builing. 
### Looking back at problem 1 where we identified the lateral load in two cases, 1: 51.69 psf & 2: 88.9 psf, we can multiply each of these by the
### trubutary square footage of 1480 to find the load carried by eack frame. For case 1 this is 76, 501 lbs and for case tw this is 131,572 lbs. In
### the next question , 4, will will use case two since it is the higher of the two cases to ensure the frame does not fail. 

In [ ]:
## Apply this lateral wind force from P3 to your braced frame computer model.
## Calculate the braced frame lateral deflection from the design wind event using your computer progra

In [11]:
# 1. Properties (Steel E and Area in inches)
E = 29e6 
area_main = np.pi * (5**2)   # 10in diameter vertical columns

# Updated Diagonals: 4in wide x 1in thick
width_diag = 4.0
thick_diag = 1.0
area_diag = width_diag * thick_diag  # Rectangular Area = 4.0 sq in

# 2. Nodes (24ft wide by 12ft tall, converted to inches)
nodes = np.array([
    [0, 0],          # Node 0: Bottom-Left (Pin Support)
    [0, 144],        # Node 1: Top-Left (Wind Load Applied Here)
    [288, 0],        # Node 2: Bottom-Right (Pin Support)
    [288, 144],      # Node 3: Top-Right
    [144, 72]        # Node 4: Center Junction
])

# 3. Elements and EA properties
elements = [[0, 1], [2, 3], [0, 4], [1, 4], [2, 4], [3, 4]]
properties = [E * area_main, E * area_main, E * area_diag, E * area_diag, E * area_diag, E * area_diag]

# 4. Boundary Conditions: Pinning both bottom feet to the ground
# This ensures the frame resists lateral movement as a single unit.
bcs = {0: [0, 0], 2: [0, 0]} 

# 5. Applied Forces
# Lateral wind force = 131,572 lbs
# Gravity load = 48,000 lbs
forces = {
    1: [131572, 0],   # Lateral Wind Force at top-left
    4: [0, -48000]     # Vertical Downward Force at center
}

# RUN SOLVER
U_result, P_result = universal_truss_solver(nodes, elements, properties, bcs, forces)

# --- RESULTS OUTPUT ---
max_deflection = U_result[2*1] # Horizontal displacement of Node 1
print(f"Design Wind Force: 131,572 lbs")
print(f"Total Lateral Deflection: {max_deflection:.4f} inches")


Design Wind Force: 131,572 lbs
Total Lateral Deflection: 0.5419 inches


In [12]:
## A typical deflection limit for wind events is H/400 where H is the frame height. Is the frame stiff enough to meet this limit?

In [13]:
# --- ANALYSIS & SAFETY CHECK ---

# 1. Calculate Allowable Drift (H/400)
height_inches = 144  # 12ft
allowable_drift = height_inches / 400

# 2. Extract Result from Solver
actual_drift = abs(U_result[2*1])  # ux of Top-Left Node

# 3. Simple Euler Buckling Check for 3" Diagonals
# L = sqrt(12^2 + 6^2) ft = 161 inches
L_diag = np.sqrt(144**2 + 72**2) 
I_diag = (np.pi * (3**4)) / 64  # Moment of Inertia for 3in round bar
P_critical = (np.pi**2 * E * I_diag) / (L_diag**2)

print(f"--- STRUCTURAL VERIFICATION ---")
print(f"Applied Lateral Load: 131,572 lbs")
print(f"Actual Lateral Deflection: {actual_drift:.4f} in")
print(f"Allowable Drift (H/400): {allowable_drift:.4f} in")

if actual_drift <= allowable_drift:
    print("RESULT: PASS - Deflection is within code limits.")
else:
    print("RESULT: FAIL - Excessive sway; stiffen the diagonals.")

# Check for Buckling in the most compressed member
max_compression = min(P_result) 
if abs(max_compression) > P_critical:
    print(f"WARNING: Buckling Failure! Member load ({abs(max_compression):.0f} lbs) exceeds limit ({P_critical:.0f} lbs).")
else:
    print("RESULT: Safe against elastic buckling.")

--- STRUCTURAL VERIFICATION ---
Applied Lateral Load: 131,572 lbs
Actual Lateral Deflection: 0.5419 in
Allowable Drift (H/400): 0.3600 in
RESULT: FAIL - Excessive sway; stiffen the diagonals.


In [ ]:
## Above is the answer to part 6 of this weeks homework. I did find that the frame is stiff enough however as you can see the waning ther seems to 
## be a small buckling failure. I think this may be due to the weight that I used for the building based on our calculations while visiting the 
## the building. We calculated 48,000psf and while completing this homework this number seemed very high and I think is causing this bucking. 